# v6 confirmation (self-contained) — new + REAL-attack data
Pre-reg: `docs/Preregistration_v6_Confirmation.md`. Colab/Kaggle T4. **Restart runtime once, then Run all top-to-bottom.** Data loading is inlined (no cached-module risk); screens/guards come from a fresh clone. Output `v6_scores.npz` → `Aegis/results/` → `python tools/v6_analysis.py`.

In [ ]:
# Cell 1 - FRESH clone (wipes any old copy) + setup
import os, sys, subprocess, glob, shutil
KAGGLE = os.path.isdir('/kaggle/working'); BASE = '/kaggle/working' if KAGGLE else '/content'
DEST = BASE + '/vyuha_src'
shutil.rmtree(DEST, ignore_errors=True)                      # guarantee latest code (no stale clone)
subprocess.run(['git','clone','--depth','1','https://github.com/g25ait2149/vyuha.git',DEST], check=True)
ROOT = os.path.dirname(os.path.dirname(glob.glob(DEST+'/**/vyuha/__init__.py', recursive=True)[0]))
sys.path.insert(0, ROOT); os.chdir(ROOT)
subprocess.run('pip -q install -U "bitsandbytes>=0.46.1" accelerate "transformers>=4.44" datasets scikit-learn sentencepiece "protobuf<5" huggingface_hub', shell=True)
tok=''
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; tok=UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; tok=userdata.get('HF_TOKEN')
except Exception as e: print('HF_TOKEN missing:', repr(e)[:100])
os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=tok or ''
from huggingface_hub import login
if tok: login(tok)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
OUT = BASE + '/v6'; os.makedirs(OUT, exist_ok=True); print('OUT =', OUT)

In [ ]:
# Cell 2 - access preflight
from huggingface_hub import HfApi
api, tk = HfApi(), os.environ.get('HF_TOKEN') or None
NEED = {'dataset':['walledai/AdvBench','walledai/MaliciousInstruct','allenai/wildjailbreak','TrustAIRLab/in-the-wild-jailbreak-prompts','allenai/wildguardmix','allenai/WildChat-1M'],
        'model':['Qwen/Qwen3Guard-Gen-0.6B','meta-llama/Llama-Guard-3-1B','meta-llama/Llama-Guard-3-8B','Qwen/Qwen2.5-1.5B-Instruct','microsoft/deberta-v3-small']}
bad=[]
for kind,repos in NEED.items():
    for r in repos:
        try: api.list_repo_files(r, repo_type=None if kind=='model' else 'dataset', token=tk); print('  OK  ',r)
        except Exception as e: bad.append(r); print(f'  FAIL https://huggingface.co/{"datasets/" if kind=="dataset" else ""}{r} : {type(e).__name__}')
assert not bad, f'accept terms / fix access: {bad}'

In [ ]:
# Cell 3 - DATA (inlined; robust WildJailbreak TSV parse + diagnostic). No dependency on cached modules.
import re, random, pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
WORD = re.compile(r'[a-z0-9]+')
def dedup(cands, refs, thr=0.6):
    rs=[set(WORD.findall(r.lower())) for r in refs]; kept=[]
    for c in cands:
        cs=set(WORD.findall(c.lower()))
        if not any((len(cs&r)/len(cs|r) if (cs or r) else 0)>=thr for r in rs): kept.append(c)
    return kept
adv=[p for p in load_dataset('walledai/AdvBench',split='train')['prompt'] if isinstance(p,str)]
mal=[p for p in load_dataset('walledai/MaliciousInstruct',split='train')['prompt'] if isinstance(p,str)]
# WildJailbreak eval: download raw TSV, parse with sep tab (default loader reads it as CSV -> 1 col)
fp=hf_hub_download('allenai/wildjailbreak','eval/eval.tsv',repo_type='dataset')
wjb=pd.read_csv(fp, sep='\t', keep_default_na=False, dtype=str)
print('WildJailbreak eval columns:', list(wjb.columns), '| rows', len(wjb))
pcol='adversarial' if 'adversarial' in wjb.columns else next((c for c in wjb.columns if 'adv' in c.lower()), wjb.columns[-1])
lcol=next((c for c in wjb.columns if 'type' in c.lower() or 'label' in c.lower()), None)
if lcol is not None: print('  label col', lcol, '->', dict(list(wjb[lcol].value_counts().items())[:6]))
def pick(sub): return [p for p in sub if isinstance(p,str) and p.strip()]
if lcol is None:
    wjb_harm=pick(wjb[pcol].tolist()); wjb_benign=[]
else:
    lv=wjb[lcol].astype(str)
    wjb_harm=pick(wjb[lv.str.contains('harm',case=False)][pcol].tolist())
    wjb_benign=pick(wjb[lv.str.contains('benign',case=False)][pcol].tolist())
itw=[t for t in load_dataset('TrustAIRLab/in-the-wild-jailbreak-prompts','jailbreak_2023_12_25',split='train')['prompt'] if isinstance(t,str) and len(t)>40]
random.Random(11).shuffle(itw)
beh=adv[:400]; itw_attacks=[itw[i%len(itw)]+'\n\nNow, specifically: '+beh[i] for i in range(min(400,len(beh)))]
wgm=load_dataset('allenai/wildguardmix','wildguardtrain',split='train').to_pandas()
wgm=wgm[wgm.prompt.notna() & wgm.prompt_harm_label.isin(['harmful','unharmful'])]
WGM=wgm.prompt.tolist(); WLAB=(wgm.prompt_harm_label=='harmful').astype(int).tolist()
wc,seen=[],set()
for row in load_dataset('allenai/WildChat-1M',split='train',streaming=True):
    if row.get('language')!='English' or row.get('toxic'): continue
    conv=row.get('conversation') or []
    if conv and conv[0].get('role')=='user':
        t=(conv[0].get('content') or '').strip()
        if 10<len(t)<4000 and t not in seen: seen.add(t); wc.append(t)
    if len(wc)>=2000: break
D6=dict(plain_advbench=dedup(adv,WGM[:4000]), plain_malicious=dedup(mal,WGM[:4000]),
        real_wildjailbreak=dedup(wjb_harm,WGM[:4000]), real_inthewild=itw_attacks,
        benign_wildchat=wc, benign_wjb_adv=wjb_benign[:1000], wgm_prompts=WGM, wgm_labels=WLAB)
for k in ['plain_advbench','plain_malicious','real_wildjailbreak','real_inthewild','benign_wildchat','benign_wjb_adv']:
    print(f'  {k:20s} {len(D6[k])}')

In [ ]:
# Cell 4 - build screens + expert, score every arm & benign, save (checkpointed per screen; empty arms skipped)
import numpy as np, gc, io, contextlib, torch
from vyuha.normalize.normalize import normalize
from vyuha.cascade.screens import make_screen
from vyuha.cascade.guards import load_model, forced_prefix, bucketed_scores
V=lambda xs:[normalize(t, full=True, version=2) for t in xs]
WGM, WLAB = D6['wgm_prompts'], D6['wgm_labels']
samp=np.random.RandomState(0).choice(len(WGM), min(6000,len(WGM)), replace=False)
KEYS=['plain_advbench','plain_malicious','real_wildjailbreak','real_inthewild','benign_wildchat','benign_wjb_adv']
OUTF=f'{OUT}/v6_scores.npz'; store=dict(np.load(OUTF,allow_pickle=True)) if os.path.exists(OUTF) else {}
def put(k,v): store[k]=np.asarray(v); np.savez(OUTF, **store)
for s in ['S2_tfidf','S1_qwen06','S5_lg3_1b','S3_probe','S4_deberta']:
    if f'{s}/{KEYS[-1]}' in store: print('[resume]',s,flush=True); continue
    print('=== screen',s,flush=True); score,free=make_screen(s,WGM,WLAB,V,samp)
    for k in KEYS:
        if f'{s}/{k}' in store: continue
        if not D6[k]: put(f'{s}/{k}',np.zeros(0)); print(f'  {s}/{k} EMPTY-skip',flush=True); continue
        put(f'{s}/{k}',score(D6[k])); print(f'  {s}/{k} {len(D6[k])}',flush=True)
    free(); gc.collect(); torch.cuda.empty_cache()
if f'E/{KEYS[-1]}' not in store:
    tok,m=load_model('meta-llama/Llama-Guard-3-8B',True)
    probes=[t for t in D6['plain_advbench'] if len(t)<300][:5]
    with contextlib.redirect_stdout(io.StringIO()): pre=forced_prefix(tok,m,'llamaguard',probes)
    for k in KEYS:
        if f'E/{k}' in store: continue
        if not D6[k]: put(f'E/{k}',np.zeros(0)); print(f'  E/{k} EMPTY-skip',flush=True); continue
        put(f'E/{k}',bucketed_scores(tok,m,V(D6[k]),'llamaguard',pre, progress=lambda i,n,k=k:(i%2000<8 or i>=n) and print(f'  E/{k} {i}/{n}',flush=True)))
    m=tok=None; gc.collect(); torch.cuda.empty_cache()
print('v6 done. Download', OUTF, '-> Aegis/results/ and run tools/v6_analysis.py')